# Triton Ensemble (NLI DeBERTa → XGBoost) on SageMaker

Deploy a two-stage **Triton ensemble** on Amazon SageMaker using **Inference Components**:

1. **NLI DeBERTa** — scores input text against N candidate labels via zero-shot NLI (ONNX, GPU)
2. **XGBoost** — classifies the resulting confidence vector into binary labels (ONNX, CPU)

## Architecture

```
input text → [NLI(text, label_1), NLI(text, label_2), ..., NLI(text, label_N)]
           → confidence_vector(1, N) → XGBoost → prediction
```

For each request, the client tokenizes N `(premise, hypothesis)` pairs and sends them as a single
`[N, max_seq_len]` batch to the NLI model. All N label inferences run **in parallel** in a single
GPU forward pass. The model includes baked-in postprocessing (softmax + entailment extraction) that
outputs a `(1, N)` confidence vector, which Triton's ensemble scheduler routes directly to XGBoost.

**Key design choice:** `max_batch_size=0` — Triton's ensemble scheduler propagates the batch
dimension uniformly, which doesn't support the fan-out (1 text → N pairs) / fan-in (N scores → 1
vector) pattern. Setting `max_batch_size=0` disables Triton's batch management so we control tensor
shapes explicitly.

This notebook was tested with the `conda_python3` kernel on an Amazon SageMaker notebook instance of type `g4dn` or `g5`.

## Contents

1. [Set up the environment](#Set-up-the-environment)
1. [Prepare models (ONNX export)](#Prepare-models-(ONNX-export))
1. [Prepare the Triton model repository](#Prepare-the-Triton-model-repository)
1. [Package and upload to S3](#Package-and-upload-to-S3)
1. [Deploy as Inference Component](#Deploy-as-Inference-Component)
1. [Test with a single request](#Test-with-a-single-request)
1. [Run inference and benchmark](#Run-inference-and-benchmark)
1. [Terminate endpoint and clean up](#Terminate-endpoint-and-clean-up)

## Set up the environment

Install dependencies and configure the SageMaker session and IAM role.

> **Restart the kernel after the `!pip install` cell below.** `transformers` and `onnx` both pin `protobuf`, and the running Python process holds onto the old version until the kernel restarts.

In [ ]:
# torchvision is pre-installed in some SageMaker kernels but unused here. The
# `pip install -U torch` below upgrades torch underneath it, breaking the ABI
# (transformers' image_utils.py then tries to import torchvision and crashes
# with `operator torchvision::nms does not exist`). Uninstall first.
!pip uninstall -y torchvision

!pip install -U pip boto3 "sagemaker>=3,<4" "transformers<5" sentencepiece "torch<2.6" onnx onnxscript
!pip install "tritonclient[http]" numpy protobuf
!pip install xgboost onnxmltools scikit-learn skl2onnx

In [ ]:
import boto3
import json
import numpy as np
import time

from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = boto3.Session()
region = sess.region_name

role = get_execution_role()

print(f"Region: {region}")
print(f"Role:   {role}")

In [ ]:
# Update the account ID from the example URL for your deployment region.
# See: https://aws.github.io/deep-learning-containers/reference/available_images/#region-availability
TRITON_ACCOUNT_ID = "763104351884"

base = "amazonaws.com.cn" if region.startswith("cn-") else "amazonaws.com"
triton_image_uri = (
    f"{TRITON_ACCOUNT_ID}.dkr.ecr.{region}.{base}"
    f"/sagemaker-tritonserver:24.09-py3"
)

print(f"Triton image URI: {triton_image_uri}")

In [ ]:
# Fixed endpoint name — reused across every deploy and update run
ENDPOINT_NAME = "triton-nli-ensemble"
INSTANCE_TYPE = "ml.g5.2xlarge"

## Prepare models (ONNX export)

Export the NLI DeBERTa model (with baked-in postprocessing) and train an XGBoost classifier, both to ONNX format.

**NLI DeBERTa:** Loads `cross-encoder/nli-deberta-v3-base`, wraps it with postprocessing that:
1. Extracts contradiction + entailment logits (drops neutral)
2. Applies 2-class softmax (matches HuggingFace `multi_label=True` behavior)
3. Reshapes entailment probabilities to `(1, N)` confidence vector

**XGBoost:** ⚠️ **Demo only** — trained on `sklearn.make_classification` synthetic data that has *nothing* to do with the NLI confidence vectors it receives at inference time. The prediction output of this end-to-end pipeline is effectively random. This exists to exercise the ensemble wiring, not to solve a task. To swap in your own trained classifier, see `convert_xgboost_to_onnx()` in [`workspace/export_models.py`](workspace/export_models.py).

This runs as a subprocess via `export_models.py` to avoid kernel memory issues.

In [ ]:
import subprocess
import sys

NLI_MODEL_ID = "cross-encoder/nli-deberta-v3-base"
N_LABELS = 50
MAX_SEQ_LEN = 128

# check=True so a failed step halts the cell — `!python` only prints the
# traceback and the rest of the notebook would happily run on a missing artifact.
subprocess.run([sys.executable, "workspace/export_models.py", "--workspace", "workspace", "--step", "nli"], check=True)
subprocess.run([sys.executable, "workspace/export_models.py", "--workspace", "workspace", "--step", "xgboost"], check=True)

## Prepare the Triton model repository

Build a 3-model Triton repository for the ensemble:

```
triton-serve-ensemble/
├── nli_deberta/                     # Stage 1: N (text, label) pairs → confidence vector (GPU)
│   ├── config.pbtxt
│   └── 1/
│       ├── model.onnx
│       └── model.onnx.data          # External weights (dynamo ONNX export)
├── xgboost_classifier/              # Stage 2: confidence vector → prediction (CPU)
│   ├── config.pbtxt
│   └── 1/model.onnx
└── ensemble_model/                  # Orchestrator (no model artifact)
    ├── config.pbtxt
    └── 1/
```

The `ensemble_model` uses Triton's `ensemble` platform to chain the NLI model's `confidence_vector` output into XGBoost's `float_input`.

All models use `max_batch_size: 0` — the fan-out/fan-in pattern requires explicit tensor shape management rather than Triton's automatic batch dimension.

> **Note:** PyTorch's dynamo-based `torch.onnx.export` (opset 18+) may save weights to a separate `model.onnx.data` file. Both files must be included.

In [ ]:
import subprocess
import sys

ENSEMBLE_REPO = "triton-serve-ensemble"

subprocess.run(
    [sys.executable, "workspace/build_triton_repo.py", "--workspace", "workspace", "--triton-repo", ENSEMBLE_REPO],
    check=True,
)

## Package and upload to S3

In [ ]:
!tar -C triton-serve-ensemble/ -czf model.tar.gz nli_deberta xgboost_classifier ensemble_model

sagemaker_session = Session(boto_session=sess)
bucket = sagemaker_session.default_bucket()
prefix = "triton-nli-ensemble"

s3_client = sess.client("s3")
s3_client.upload_file("model.tar.gz", bucket, f"{prefix}/model.tar.gz")
model_uri = f"s3://{bucket}/{prefix}/model.tar.gz"

print(f"Model artifact uploaded to: {model_uri}")

## Deploy as Inference Component

Deploy the Triton ensemble as an **Inference Component**. The `SAGEMAKER_TRITON_DEFAULT_MODEL_NAME` is set to `ensemble_model` so Triton routes incoming requests to the ensemble orchestrator, which internally calls both the NLI DeBERTa model and XGBoost.

The resource requirements specify the compute allocation for this component:
- `memory` — minimum memory in MB
- `num_accelerators` — number of GPU devices (NLI DeBERTa runs on GPU)
- `num_cpus` — number of CPU cores (XGBoost runs on CPU)
- `copies` — number of model copies to run

In [ ]:
from sagemaker.serve.model_builder import ModelBuilder, ModelServer
from sagemaker.core.inference_config import ResourceRequirements
from sagemaker.core.resources import Endpoint, InferenceComponent

ENSEMBLE_MODEL_NAME = "ensemble_model"
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
sm_model_name = f"triton-nli-ensemble-{timestamp}"
ic_name = f"triton-nli-ensemble-ic-{timestamp}"

# Naming convention:
#   - ENDPOINT_NAME and the endpoint config: stable across runs.
#   - sm_model_name and ic_name: timestamped per deploy.
#
# Why keep the endpoint name stable?
#      Faster iteration — once the endpoint is up, redeploys only need to
#      swap the IC. We skip the multi-minute instance provisioning + container
#      pull and just attach a new IC to the existing instance.
#
# If the endpoint is already InService, delete the old IC to free the GPU
# before attaching the new one (ml.g5.2xlarge has a single GPU, so two ICs
# can't coexist).
sm_client = boto3.client("sagemaker")

try:
    sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    existing_ics = sm_client.list_inference_components(EndpointNameEquals=ENDPOINT_NAME)["InferenceComponents"]
    for old_ic in existing_ics:
        old_name = old_ic["InferenceComponentName"]
        print(f"Deleting existing IC '{old_name}' to free the GPU for the new deploy")
        sm_client.delete_inference_component(InferenceComponentName=old_name)
        while True:
            try:
                sm_client.describe_inference_component(InferenceComponentName=old_name)
                time.sleep(5)
            except sm_client.exceptions.ClientError:
                break
except sm_client.exceptions.ClientError:
    pass  # endpoint doesn't exist — ModelBuilder will create it fresh

builder = ModelBuilder(
    image_uri=triton_image_uri,
    s3_model_data_url=model_uri,
    role_arn=role,
    env_vars={"SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": ENSEMBLE_MODEL_NAME},
    model_server=ModelServer.TRITON,
    instance_type=INSTANCE_TYPE,
    sagemaker_session=sagemaker_session,
)
builder.build(model_name=sm_model_name)
print(f"SM model created: {sm_model_name}")

print(f"\nDeploying inference component '{ic_name}' ...")
builder.deploy(
    endpoint_name=ENDPOINT_NAME,
    inference_component_name=ic_name,
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    inference_config=ResourceRequirements(
        requests={"memory": 8192, "num_accelerators": 1, "num_cpus": 2, "copies": 1}
    ),
    wait=False,
)

# Brief pause so the endpoint resource is queryable before we poll for status
time.sleep(3)

Endpoint.get(ENDPOINT_NAME).wait_for_status("InService")
ic = InferenceComponent.get(ic_name)
ic.wait_for_status("InService")
print(f"\nEndpoint:            {ENDPOINT_NAME}")
print(f"Inference component: {ic_name}")

## Test with a single request

Before benchmarking, invoke the endpoint once to confirm it's wired up and to see the raw request/response shape. This is the minimal integration pattern you'd drop into your own application:

1. **Tokenize** the input text against all N candidate labels → `[N, max_seq_len]` tensors
2. **Send** the tensors as a Triton HTTP payload with `InferenceComponentName` set
3. **Parse** the response — the `label` output is the XGBoost binary prediction

> ⚠️ The `Prediction` value below is meaningless — the XGBoost stage was trained on `sklearn.make_classification` synthetic data, not on real NLI confidence vectors. We're exercising the ensemble wiring end to end, not solving a task. Swap in your own trained classifier (see `convert_xgboost_to_onnx()` in [`workspace/export_models.py`](workspace/export_models.py)) for real predictions.

In [ ]:
from transformers import AutoTokenizer

from workspace.config import NLI_MODEL_ID, NLI_LABELS, HYPOTHESIS_TEMPLATE

# 1. Tokenize text against all 50 labels (client-side)
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_ID)
text = "Your Office365 password expires today. Click here to verify your credentials and avoid account lockout."

premises = [text] * N_LABELS
hypotheses = [HYPOTHESIS_TEMPLATE.format(label) for label in NLI_LABELS]
encoded = tokenizer(
    premises, hypotheses,
    padding="max_length", truncation=True, max_length=MAX_SEQ_LEN,
    return_tensors="np",
)

# 2. Build the Triton HTTP payload and invoke
payload = {
    "inputs": [
        {"name": "input_ids",      "shape": [N_LABELS, MAX_SEQ_LEN], "datatype": "INT64", "data": encoded["input_ids"].tolist()},
        {"name": "attention_mask", "shape": [N_LABELS, MAX_SEQ_LEN], "datatype": "INT64", "data": encoded["attention_mask"].tolist()},
    ]
}

runtime = boto3.client("sagemaker-runtime", region_name=region)
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    InferenceComponentName=ic_name,
    ContentType="application/octet-stream",
    Body=json.dumps(payload),
)

# 3. Parse the response — the ensemble's single output is the XGBoost label
result = json.loads(response["Body"].read().decode("utf-8"))
prediction = result["outputs"][0]["data"][0]
print(f"Input:      {text}")
print(f"Prediction: {prediction}")
print(f"Raw output: {json.dumps(result, indent=2)[:400]}...")

## Run inference and benchmark

Once the endpoint is in service, we benchmark it using `run_benchmark.py`. The script:

- Constructs 50 `(text, hypothesis)` pairs per request and tokenizes them client-side
- Sends `[50, 128]` tensors to the Triton ensemble via `InferenceComponentName`
- All 50 NLI label inferences run in parallel in a single GPU forward pass
- Reports latency statistics (min, mean, median, p90, p95, p99, max) and sample predictions

In [ ]:
!python workspace/run_benchmark.py \
    --endpoint-name {ENDPOINT_NAME} \
    --inference-component-name {ic_name} \
    --warmup 5 \
    --iterations 50

## Terminate endpoint and clean up

Delete the inference component, endpoint, endpoint configuration, and model to avoid ongoing charges.

Resources are deleted in dependency order: inference component first, then endpoint, endpoint config, and finally the model.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model, InferenceComponent

# Delete inference component
ic = InferenceComponent.get(ic_name)
ic.delete()
print(f"Deleted inference component: {ic_name}")

# Wait for IC deletion before deleting endpoint
time.sleep(30)

# Delete endpoint
ep = Endpoint.get(ENDPOINT_NAME)
ep.delete()
print(f"Deleted endpoint: {ENDPOINT_NAME}")

# Wait for endpoint deletion before deleting config
time.sleep(30)

# Delete endpoint config
epc = EndpointConfig.get(ep.endpoint_config_name)
epc.delete()
print(f"Deleted endpoint config: {ep.endpoint_config_name}")

# Delete model
mdl = Model.get(sm_model_name)
mdl.delete()
print(f"Deleted model: {sm_model_name}")

print("\nCleanup complete.")

In [ ]:
# Optional: clean up local artifacts
!rm -rf triton-serve-ensemble/
!rm -rf workspace/nli_deberta workspace/xgboost
!rm -f model.tar.gz
print("Local artifacts cleaned up.")